# Problem 879 - Touch-screen Password
A touch-screen device can be unlocked with a "password" consisting of a sequence of two or more distinct spots that the user selects from a rectangular grid of spots on the screen. The user enters their sequence by touching the first spot, then tracing a straight line segment to the next spot, and so on until the end of the sequence. The user's finger remains in contact with the screen throughout, and may only move in straight line segments from spot to spot.

If the finger traces a straight line that passes over an intermediate spot, then that is treated as two line segments with the intermediate spot included in the password sequence. For example, on a $3\times 3$ grid labelled with digits $1$ to $9$ (shown below), tracing $1-9$ is interpreted as $1-5-9$.

Once a spot has been selected it disappears from the screen. Thereafter, the spot may not be used as an endpoint of future line segments, and it is ignored by any future line segments which happen to pass through it. For example, tracing $1-9-3-7$ (which crosses the $5$ spot twice) will give the password $1-5-9-6-3-7$.

There are $389488$ different passwords that can be formed on a $3 \times 3$ grid.

Find the number of different passwords that can be formed on a $4 \times 4$ grid.

## Solution.

In [1]:
import numpy as np
from functools import cache

In [2]:
def rotate(A):
    n = A.shape[0]
    
    B = np.zeros((n, n))
    for i in range(n):
        B[i, n-1-i] = 1

    return A.T @ B


def reflection(A):
    # reflect left-righ
    n = A.shape[0]

    B = np.zeros((n, n))
    for i in range(n):
        B[i, n-1-i] = 1

    return A @ B


def equivalence_class(A):
    ans = set()
    
    def add_matrix(M):
        ans.add(tuple(map(tuple, M)))

    # rotations
    R = A
    for _ in range(4):
        add_matrix(R)
        R = rotate(R)

    # reflected rotations
    R = reflection(A)
    for _ in range(4):
        add_matrix(R)
        R = rotate(R)

    return [np.array(M) for M in ans]

In [3]:
import numpy as np

def reachable(A, x, y):
    n = A.shape[0]
    ans = []

    for i in range(n):
        for j in range(n):
            # skip empty cells and the starting point
            if A[i, j] != 1 or (i == x and j == y):
                continue

            blocked = False

            # vertical line
            if i == x:
                for k in range(min(j, y) + 1, max(j, y)):
                    if A[i, k] == 1:
                        blocked = True
                        break

            # horizontal line
            elif j == y:
                for k in range(min(i, x) + 1, max(i, x)):
                    if A[k, j] == 1:
                        blocked = True
                        break

            # general line
            else:
                dx = i - x
                dy = j - y

                for k in range(min(i, x) + 1, max(i, x)):
                    # check whether this x-coordinate gives an integer y-coordinate
                    num = dy * (k - x)

                    if num % dx == 0:
                        l = y + num // dx

                        if A[k, l] == 1:
                            blocked = True
                            break

            if not blocked:
                ans.append((i, j))

    return ans

In [4]:
@cache
def dp(A, i, j):
    # A is a hashable tuple-of-tuples
    An = np.array(A)

    if An.sum() == 0:
        return 1

    neighbours = reachable(An, i, j)

    boards = {}
    for x, y in neighbours:
        B = An.copy()
        B[x, y] = 0
        boards[(x, y)] = tuple(map(tuple, B))

    seen = set()
    ans = 1   # stop the password here

    for x, y in neighbours:
        B = An.copy()
        B[x,y] = 0
        ans += dp(tuple(map(tuple,B)), x, y)
        
    return ans

In [6]:
def solv(n):
    A = np.ones((n, n))

    ans = 0
    seen = set()

    for i in range(n):
        for j in range(n):
            B = A.copy()
            B[i, j] = 0
            key = tuple(map(tuple, B))
            EC = equivalence_class(B)

            if key not in seen:
                ans += dp(key, i, j) * len(EC)


                for M in EC:
                    seen.add(tuple(map(tuple, M)))

    return ans - n*n # remove length 1 password

In [7]:
solv(3)

389488

In [ ]:
solv(4)  == 

4350069824940